<a href="https://colab.research.google.com/github/fiserjn-ship-it/Bohemiancoin-v2/blob/python_prototype/Bohemiancoin_0_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import hashlib
import time
import math
import random
from dataclasses import dataclass, field
from typing import List, Dict

# --- KONSTANTY SYSTÉMU ---
CODEX_SIZE_MB = 512
MAX_TRANSACTIONS_PER_BLOCK = 30

# --- POMOCNÁ FUNKCE PRO HASH (SHA-256 místo BLAKE3 pro kompatibilitu) ---
def get_hash(data: str):
    return hashlib.sha256(data.encode()).hexdigest()

# --- DATOVÉ STRUKTURY ---

@dataclass
class Chronos:
    epocha: int
    codex_seq: int
    actum_index: int

    def __str__(self):
        return f"{self.epocha}:{self.codex_seq}:{self.actum_index}"

@dataclass
class Actum:
    sender_id: int
    receiver_id: int
    amount: float
    fee: float
    nonce: int
    signature: str = "valid_sig"
    status: str = "Active"

    def to_hash(self):
        data = f"{self.sender_id}{self.receiver_id}{self.amount}{self.fee}{self.nonce}"
        return get_hash(data)

class BohemianNetwork:
    def __init__(self):
        # Simulace Codexu (1000 uživatelů)
        self.codex = {i: {"balance": 1000.0, "nonce": 0, "status": "Green"} for i in range(1000)}
        self.virbitirium: List[Actum] = []
        self.last_block_hash = "0" * 64
        self.current_chronos = Chronos(1, 1, 0)

    def add_to_virbitirium(self, actum: Actum):
        user_slot = self.codex[actum.sender_id]
        if user_slot["balance"] < actum.amount:
            print(f"Zamítnuto: Nedostatek prostředků u uživatele {actum.sender_id}")
            return False

        # Přesun do Šedé zóny
        user_slot["status"] = "Pendens"
        self.virbitirium.append(actum)
        print(f"Actum přidáno do Virbitiria. Uživatel {actum.sender_id} je v Šedé zóně.")
        return True

    def select_manifest(self) -> List[Actum]:
        """Pravidlo 15:10:5"""
        sorted_pool = sorted(self.virbitirium, key=lambda x: x.fee, reverse=True)
        priority = sorted_pool[:15]
        waiting = sorted_pool[15:25]

        manifest = priority + waiting
        while len(manifest) < MAX_TRANSACTIONS_PER_BLOCK:
            manifest.append(Actum(-1, -1, 0, 0, 0, "NULL_TX"))
        return manifest

    def the_cubic_seal(self, manifest: List[Actum], hunter_id: str):
        """Geometrický Proof-of-Physical-Read"""
        # 1. Výpočet Bodu T (Cíl)
        manifest_hash = get_hash(str(manifest))
        t_data = f"{self.last_block_hash}{self.current_chronos}{hunter_id}{manifest_hash}"
        t_hash_int = int(get_hash(t_data), 16)

        # Souřadnice v 3D prostoru (zjednodušeno na 2**85 rozsah)
        xt = t_hash_int % (2**85)
        yt = (t_hash_int >> 85) % (2**85)
        zt = (t_hash_int >> 170) % (2**85)

        print(f"Lovec {hunter_id} zaměřuje Bod T: [{xt}, {yt}, {zt}]")

        best_d = float('inf')
        winning_nonce = 0

        # Simulace 1000 pokusů GPU o nalezení blízkého bodu
        for _ in range(1000):
            nonce = random.randint(0, 10**10)
            p_hash_int = int(get_hash(str(nonce)), 16)
            xp, yp, zp = p_hash_int % (2**85), (p_hash_int >> 85) % (2**85), (p_hash_int >> 170) % (2**85)

            d = math.sqrt((xp-xt)**2 + (yp-yt)**2 + (zp-zt)**2)
            if d < best_d:
                best_d = d
                winning_nonce = nonce

        print(f"Nejlepší nalezená vzdálenost d: {best_d:.2e}")
        return winning_nonce, best_d

    def process_block(self, manifest: List[Actum]):
        """Zápis do Codexu a aktualizace sítě"""
        processed_count = 0
        for tx in manifest:
            if tx.sender_id != -1:
                self.codex[tx.sender_id]["balance"] -= (tx.amount + tx.fee)
                self.codex[tx.receiver_id]["balance"] += tx.amount
                self.codex[tx.sender_id]["status"] = "Green"
                self.codex[tx.sender_id]["nonce"] += 1
                if tx in self.virbitirium:
                    self.virbitirium.remove(tx)
                processed_count += 1

        self.current_chronos.codex_seq += 1
        self.last_block_hash = get_hash(str(time.time()))
        print(f"Blok potvrzen! Zpracováno {processed_count} transakcí. Nový Chronos: {self.current_chronos}")

# --- TESTOVACÍ SCÉNÁŘ ---

# 1. Inicializace sítě
bc = BohemianNetwork()

# 2. Vytvoření transakce (Uživatel 500 posílá 50 GRS uživateli 100)
print("--- KROK 1: ODESLÁNÍ ---")
tx1 = Actum(sender_id=500, receiver_id=100, amount=50.0, fee=1.5, nonce=0)
bc.add_to_virbitirium(tx1)

# 3. Výběr manifestu (Lovci berou transakce z Virbitiria)
print("\n--- KROK 2: MANIFEST ---")
manifest = bc.select_manifest()

# 4. Výpočet Cubic Seal (Lovecký proces)
print("\n--- KROK 3: THE CUBIC SEAL ---")
nonce, distance = bc.the_cubic_seal(manifest, "Hunter_Alpha")

# 5. Potvrzení bloku
print("\n--- KROK 4: ZÁPIS DO CODEXU ---")
bc.process_block(manifest)

# 6. Kontrola výsledku
print(f"\nKonečný zůstatek odesílatele (500): {bc.codex[500]['balance']} GRS")
print(f"Konečný zůstatek příjemce (100): {bc.codex[100]['balance']} GRS")
print(f"Stav slotu uživatele 500: {bc.codex[500]['status']}")

--- KROK 1: ODESLÁNÍ ---
Actum přidáno do Virbitiria. Uživatel 500 je v Šedé zóně.

--- KROK 2: MANIFEST ---

--- KROK 3: THE CUBIC SEAL ---
Lovec Hunter_Alpha zaměřuje Bod T: [25950938382605575358896366, 35186979478467397900949900, 13310691976736095658802880]
Nejlepší nalezená vzdálenost d: 2.64e+24

--- KROK 4: ZÁPIS DO CODEXU ---
Blok potvrzen! Zpracováno 1 transakcí. Nový Chronos: 1:2:0

Konečný zůstatek odesílatele (500): 948.5 GRS
Konečný zůstatek příjemce (100): 1050.0 GRS
Stav slotu uživatele 500: Green
